In [4]:
import models as mm
import customDatasets
import time
import csv
import matplotlib.pyplot as plt
import json
import torch
import math
import numpy as np
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import json

In [5]:
# Local data location
test_indices = [0]
microscope_imgs_dir_path = "../images/microscope"
labels_file_path = "./labelsinfo.csv"
label_type = "om-regression"
model_weights_path = "../weights/microscope.zip"

# Define model and training configurations
microscope_img_roi = 1920
img_divisions_n = 2

microscope_img_channels = 3
microscope_height_resize = 224
microscope_width_resize = 224

rotations = [0]

# Loaders batch size
batch_size = 16

# Models Params
model_output_dims = {
    "om-regression": 4,
    "mineral-regression": 3
}
model_type = "tiny"
pre_trained = False
train_only_last_layer = False

# Build cropboxes
microscope_sub_img_dim = microscope_img_roi // img_divisions_n

microscope_img_left = 0
microscope_img_upper = 0
microscope_img_right = microscope_sub_img_dim
microscope_img_lower = microscope_sub_img_dim

# Select one quadrant per picture
microscope_cropboxes = [(microscope_img_left, microscope_img_upper, microscope_img_right, microscope_img_lower)] # (left, upper, right, lower)
        
# Select all 4 quadrants per picture
# microscope_cropboxes = customDatasets.createCropBoxes(img_divisions_n, microscope_sub_img_dim)

# Transforms
custome_normalization_transform = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

test_dataset = customDatasets.OneImageCropboxRotationDataset(
    microscope_imgs_dir_path,
    labels_file_path,
    label_type,
    test_indices,
    microscope_cropboxes,
    microscope_img_channels,
    microscope_height_resize,
    microscope_width_resize,
    rotations_values=rotations,
    transform=custome_normalization_transform,
)

test_dataset_n = test_dataset.__len__()

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
)

# Build model
model = mm.ConvnextTrainer.instantiate_model(_, model_type, label_type, False, train_only_last_layer)

model.load_state_dict(torch.load(model_weights_path, map_location=torch.device('cpu')))
model.eval()

batch_counter = 0
total_iterations = math.ceil(test_dataset_n / batch_size)

test_labels = []
test_raw_preds = []

for microscope_inputs, labels in test_loader:
    batch_counter += 1
    print("Batch: {}/{}".format(batch_counter, total_iterations))

    # Make predictions for this batch
    raw_preds = model(microscope_inputs)

    # Save predictions and labels
    test_labels.append(labels.numpy())
    test_raw_preds.append(raw_preds.detach().numpy())

predictions = np.concatenate(test_raw_preds, axis=0)
labels = np.concatenate(test_labels, axis=0)

0 samples processed
Batch: 1/1


In [6]:
predictions

array([[0.60607237, 0.25497076, 0.08885162, 0.05010519]], dtype=float32)

In [8]:
mse = np.mean((labels - predictions) ** 2, axis=0)
print("Per feature MSE:", mse)

overall_mse = np.mean(mse)
print("Overall MSE:", overall_mse)

Per feature MSE: [0.07575394 0.04969571 0.00018347 0.00158119]
Overall MSE: 0.03180358
